# Kaggle 2×T4 correctness notebook — fast async run

Цель ноутбука: быстро проверить реальную логику multi-GPU beam search без правок C++/CUDA:

- загрузка `puzzle_info.json` и `test.csv`;
- сборка CUDA extension;
- 1-GPU correctness: central/depth1/depth2;
- 2-GPU correctness: NCCL/stream3 path, `remote_packed > 0`;
- CUDA Graph включён через `USE_CUDA_GRAPHS=1`;
- TorchScript MLP ensemble выключен в быстром прогоне, чтобы не смешивать проверку stream-pipeline и trace/export MLP.

Notebook-level logging: subprocess запускается через live-wrapper с heartbeat и `nvidia-smi` snapshot. C++/CUDA код не меняется.


In [1]:
import os
import sys
import json
import time
import pathlib
import shutil
import subprocess
import threading
import queue
from datetime import datetime

import torch

print('python', sys.version)
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
print('cuda_device_count', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'gpu={i}; name={p.name}; total_memory_GiB={p.total_memory/1024**3:.2f}; capability={p.major}.{p.minor}')


python 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch 2.10.0+cu128
cuda_available True
cuda_device_count 2
gpu=0; name=Tesla T4; total_memory_GiB=14.56; capability=7.5
gpu=1; name=Tesla T4; total_memory_GiB=14.56; capability=7.5


In [2]:
PROJECT_DIR = pathlib.Path('/kaggle/working/CayleyBeam100H100')

if not PROJECT_DIR.exists():
    input_root = pathlib.Path('/kaggle/input')
    input_candidates = []
    if input_root.exists():
        input_candidates = [p for p in input_root.rglob('*') if p.is_dir() and (p / 'beam_engine.py').exists()]
    if input_candidates:
        shutil.copytree(input_candidates[0], PROJECT_DIR)
        print('copied_from_input', input_candidates[0], '->', PROJECT_DIR)

if not PROJECT_DIR.exists():
    input_root = pathlib.Path('/kaggle/input')
    zip_candidates = []
    if input_root.exists():
        zip_candidates = [p for p in input_root.rglob('source.zip') if p.is_file()]
    for zip_path in zip_candidates:
        shutil.unpack_archive(str(zip_path), PROJECT_DIR)
        print('unpacked_from_input_zip', zip_path, '->', PROJECT_DIR)
        if (PROJECT_DIR / 'beam_engine.py').exists():
            break
        shutil.rmtree(PROJECT_DIR, ignore_errors=True)

if not PROJECT_DIR.exists():
    cwd_candidates = [p for p in pathlib.Path('.').resolve().rglob('*') if p.is_dir() and (p / 'beam_engine.py').exists()]
    if cwd_candidates:
        PROJECT_DIR = cwd_candidates[0]

assert PROJECT_DIR.exists(), 'Project directory with beam_engine.py not found'
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('PROJECT_DIR', PROJECT_DIR)
print('files_ok', (PROJECT_DIR / 'beam_engine.py').exists(), (PROJECT_DIR / 'beam_engine.cpp').exists(), (PROJECT_DIR / 'beam_kernels.cu').exists())


copied_from_input /kaggle/input/datasets/trydotatwo/kekaff -> /kaggle/working/CayleyBeam100H100
PROJECT_DIR /kaggle/working/CayleyBeam100H100
files_ok True True True


In [3]:
def now() -> str:
    return datetime.now().strftime('%H:%M:%S')


def banner(title: str) -> None:
    print('\n' + '=' * 96, flush=True)
    print(f'[{now()}] {title}', flush=True)
    print('=' * 96, flush=True)


def gpu_snapshot() -> str:
    try:
        out = subprocess.check_output(
            [
                'nvidia-smi',
                '--query-gpu=index,name,memory.used,memory.total,utilization.gpu',
                '--format=csv,noheader,nounits',
            ],
            text=True,
            stderr=subprocess.STDOUT,
        ).strip()
        return out.replace('\n', ' | ')
    except Exception as e:
        return f'nvidia-smi-unavailable: {type(e).__name__}: {e}'


def _reader_thread(pipe, q: queue.Queue):
    try:
        for line in iter(pipe.readline, ''):
            q.put(line.rstrip('\n'))
    finally:
        try:
            pipe.close()
        except Exception:
            pass


def run_live(cmd, *, env=None, cwd=None, title=None, heartbeat_sec=15, timeout_sec=None):
    # Notebook-side live subprocess wrapper.
    # No torch.cuda.synchronize. No C++/CUDA file modifications.
    if title:
        banner(title)
    print(f'[{now()}] cmd={cmd}', flush=True)
    print(f'[{now()}] cwd={cwd or os.getcwd()}', flush=True)

    merged_env = os.environ.copy()
    if env:
        merged_env.update({str(k): str(v) for k, v in env.items()})
    merged_env.setdefault('PYTHONUNBUFFERED', '1')

    start = time.time()
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd or os.getcwd()),
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    q = queue.Queue()
    t = threading.Thread(target=_reader_thread, args=(proc.stdout, q), daemon=True)
    t.start()

    lines = []
    last_heartbeat = 0.0
    last_output_time = time.time()

    while True:
        drained = False
        while True:
            try:
                line = q.get_nowait()
            except queue.Empty:
                break
            drained = True
            last_output_time = time.time()
            lines.append(line)
            print(line, flush=True)

        rc = proc.poll()
        elapsed = time.time() - start

        if timeout_sec is not None and elapsed > timeout_sec and rc is None:
            proc.kill()
            raise TimeoutError(f'command timeout after {elapsed:.1f}s: {cmd}')

        if rc is not None:
            while True:
                try:
                    line = q.get_nowait()
                except queue.Empty:
                    break
                lines.append(line)
                print(line, flush=True)
            print(f'[{now()}] process_exit | returncode={rc} | elapsed_sec={elapsed:.1f}', flush=True)
            if rc != 0:
                raise subprocess.CalledProcessError(rc, cmd, output='\n'.join(lines))
            return '\n'.join(lines)

        if (time.time() - last_heartbeat) >= heartbeat_sec:
            silent_for = time.time() - last_output_time
            print(
                f'[{now()}] still_running | elapsed_sec={elapsed:.1f} | silent_for_sec={silent_for:.1f} | gpu=[{gpu_snapshot()}]',
                flush=True,
            )
            last_heartbeat = time.time()

        time.sleep(0.2 if drained else 0.5)


In [4]:
FAST_ENV = {
    # execution
    'PYTHONUNBUFFERED': '1',

    # mandatory graph path
    'USE_CUDA_GRAPHS': '1',

    # fast correctness backend; MLP is tested separately after stream correctness
    'INFERENCE_BACKEND': 'central_hamming',
    'TEST_TORCHSCRIPT_ENSEMBLE': '0',

    # small but nontrivial sizes for Kaggle T4 debug
    'GLOBAL_BEAM_WIDTH': '4096',
    'B_MICRO': '512',
    'SCORE_RING_DEPTH': '4',
    'NET_RING_DEPTH': '2',
    'BUCKET_CAP_PER_PEER': '8192',
    'K_EXPAND_TILE': '8192',
    'INFERENCE_PARALLELISM': '2',
    'MAX_DEPTH': '3',
    'HISTOGRAM_PERIOD_MICRO': '2',

    # quiet runtime; notebook wrapper prints heartbeat
    'LOG_LEVEL': 'WARNING',
    'BUILD_VERBOSE': '0',
    'EXT_VERBOSE': '0',

    # Kaggle local NCCL
    'NCCL_IB_DISABLE': '1',
    'NCCL_P2P_DISABLE': '1',
    'NCCL_SOCKET_IFNAME': 'lo',
    'GLOO_SOCKET_IFNAME': 'lo',
    'NCCL_DEBUG': 'WARN',
}

print(json.dumps(FAST_ENV, indent=2, ensure_ascii=False))


{
  "PYTHONUNBUFFERED": "1",
  "USE_CUDA_GRAPHS": "1",
  "INFERENCE_BACKEND": "central_hamming",
  "TEST_TORCHSCRIPT_ENSEMBLE": "0",
  "GLOBAL_BEAM_WIDTH": "4096",
  "B_MICRO": "512",
  "SCORE_RING_DEPTH": "4",
  "NET_RING_DEPTH": "2",
  "BUCKET_CAP_PER_PEER": "8192",
  "INFERENCE_PARALLELISM": "2",
  "MAX_DEPTH": "3",
  "HISTOGRAM_PERIOD_MICRO": "2",
  "LOG_LEVEL": "WARNING",
  "BUILD_VERBOSE": "0",
  "EXT_VERBOSE": "0",
  "NCCL_IB_DISABLE": "1",
  "NCCL_P2P_DISABLE": "1",
  "NCCL_SOCKET_IFNAME": "lo",
  "GLOO_SOCKET_IFNAME": "lo",
  "NCCL_DEBUG": "WARN"
}


In [5]:
banner('Python-side data loader correctness')
import data_loader

# No GPU synchronization here. Pure CPU validation.
data_loader.validate_inverse_pairs()
central = data_loader.get_central_state_u8()
tests = data_loader.load_test_puzzles(max_puzzles=3)
print('central_shape', central.shape)
print('central_first10', central[:10].tolist())
print('actions', data_loader.ACTION_NAMES)
print('test_rows_loaded', len(tests))
print('first_test_id', tests[0][0])
print('first_test_first10', tests[0][1][:10].tolist())
print('owner_U_world2', data_loader.owner_rank_for_state(data_loader.apply_action_cpu(central, 'U'), 2))
assert len(tests) > 0
print('data_loader_ok=True')



[15:57:59] Python-side data loader correctness
central_shape (120,)
central_first10 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
actions ['-B', '-BL', '-BR', '-D', '-DL', '-DR', '-F', '-FL', '-FR', '-L', '-R', '-U', 'B', 'BL', 'BR', 'D', 'DL', 'DR', 'F', 'FL', 'FR', 'L', 'R', 'U']
test_rows_loaded 3
first_test_id 0
first_test_first10 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
owner_U_world2 1
data_loader_ok=True


In [6]:
# Memory sizing. This is CPU-only arithmetic and does not touch engine internals.
run_live(
    [sys.executable, '-u', 'scripts/t4_sizing.py'],
    env=FAST_ENV,
    cwd=PROJECT_DIR,
    title='T4 memory sizing for fast correctness config',
    heartbeat_sec=10,
    timeout_sec=120,
)



[15:57:59] T4 memory sizing for fast correctness config
[15:57:59] cmd=['/usr/bin/python3', '-u', 'scripts/t4_sizing.py']
[15:57:59] cwd=/kaggle/working/CayleyBeam100H100
[15:57:59] still_running | elapsed_sec=0.0 | silent_for_sec=0.0 | gpu=[0, Tesla T4, 3, 15360, 0 | 1, Tesla T4, 3, 15360, 0]
entity_id=t4_sizing; type=memory_model; state=calculated
params: WORLD_SIZE=2; GLOBAL_BEAM_WIDTH=4096; B_MICRO=512; FANOUT=24; BUCKET_CAP_PER_PEER=8192; INFERENCE_PARALLELISM=2
derived: N_LOCAL=2048; K_KEEP=2150; K_WORK=2473; HASH_CAPACITY=4496
buffer=beam_current; bytes=245760; GiB=0.000
buffer=next_state_pool; bytes=296760; GiB=0.000
buffer=next_meta; bytes=79136; GiB=0.000
buffer=hash_table; bytes=143872; GiB=0.000
buffer=current_active_flags; bytes=2048; GiB=0.000
buffer=active_flags; bytes=2473; GiB=0.000
buffer=score_ring; bytes=98304; GiB=0.000
buffer=send_buckets; bytes=5242880; GiB=0.005
buffer=recv_buckets; bytes=5242880; GiB=0.005
buffer=send_counts; bytes=16; GiB=0.000
buffer=histogr

'entity_id=t4_sizing; type=memory_model; state=calculated\nparams: WORLD_SIZE=2; GLOBAL_BEAM_WIDTH=4096; B_MICRO=512; FANOUT=24; BUCKET_CAP_PER_PEER=8192; INFERENCE_PARALLELISM=2\nderived: N_LOCAL=2048; K_KEEP=2150; K_WORK=2473; HASH_CAPACITY=4496\nbuffer=beam_current; bytes=245760; GiB=0.000\nbuffer=next_state_pool; bytes=296760; GiB=0.000\nbuffer=next_meta; bytes=79136; GiB=0.000\nbuffer=hash_table; bytes=143872; GiB=0.000\nbuffer=current_active_flags; bytes=2048; GiB=0.000\nbuffer=active_flags; bytes=2473; GiB=0.000\nbuffer=score_ring; bytes=98304; GiB=0.000\nbuffer=send_buckets; bytes=5242880; GiB=0.005\nbuffer=recv_buckets; bytes=5242880; GiB=0.005\nbuffer=send_counts; bytes=16; GiB=0.000\nbuffer=histograms_threshold_counters_status; bytes=524360; GiB=0.000\ntotal_static_buffers_bytes=11878489; total_static_buffers_GiB=0.011\nt4_15gb_headroom_GiB=14.989; memory_ok=True\nnote=Kaggle runtime overhead, CUDA context, NCCL internals, TorchScript model weights, inference output tensors,

In [7]:
# Build-only check. First run can be slow because nvcc/c++ compile the extension.
# Notebook heartbeat confirms progress without changing C++/CUDA.
build_env = FAST_ENV.copy()
build_env.update({'WORLD_SIZE': '1', 'RANK': '0', 'LOCAL_RANK': '0', 'CUDA_VISIBLE_DEVICES': '0'})

run_live(
    [
        sys.executable,
        '-u',
        '-c',
        "from beam_engine import build_extension; ext = build_extension(verbose=False); print('extension_built=True'); print(ext)",
    ],
    env=build_env,
    cwd=PROJECT_DIR,
    title='Build CUDA extension',
    heartbeat_sec=15,
    timeout_sec=900,
)



[15:58:00] Build CUDA extension
[15:58:00] cmd=['/usr/bin/python3', '-u', '-c', "from beam_engine import build_extension; ext = build_extension(verbose=False); print('extension_built=True'); print(ext)"]
[15:58:00] cwd=/kaggle/working/CayleyBeam100H100
[15:58:00] still_running | elapsed_sec=0.0 | silent_for_sec=0.0 | gpu=[0, Tesla T4, 3, 15360, 0 | 1, Tesla T4, 3, 15360, 0]
[15:58:15] still_running | elapsed_sec=15.1 | silent_for_sec=15.0 | gpu=[0, Tesla T4, 3, 15360, 0 | 1, Tesla T4, 3, 15360, 0]
[15:58:30] still_running | elapsed_sec=30.1 | silent_for_sec=30.1 | gpu=[0, Tesla T4, 3, 15360, 0 | 1, Tesla T4, 3, 15360, 0]
extension_built=True
<module 'beam_engine_ext' from '/root/.cache/torch_extensions/py312_cu128/beam_engine_ext/beam_engine_ext.so'>
[15:58:43] process_exit | returncode=0 | elapsed_sec=42.9


"extension_built=True\n<module 'beam_engine_ext' from '/root/.cache/torch_extensions/py312_cu128/beam_engine_ext/beam_engine_ext.so'>"

In [8]:
# 1-GPU correctness.
# Expected: final JSON contains cases depth0/depth1/depth2 and cuda_graph_captured=true.
one_rank_env = FAST_ENV.copy()
one_rank_env.update({
    'WORLD_SIZE': '1',
    'RANK': '0',
    'LOCAL_RANK': '0',
    'CUDA_VISIBLE_DEVICES': '0',
})

out_1gpu = run_live(
    [sys.executable, '-u', 'scripts/kaggle_correctness_check.py'],
    env=one_rank_env,
    cwd=PROJECT_DIR,
    title='1-GPU correctness: CUDA Graph + stop-on-solved/max-depth',
    heartbeat_sec=15,
    timeout_sec=600,
)



[15:58:43] 1-GPU correctness: CUDA Graph + stop-on-solved/max-depth
[15:58:43] cmd=['/usr/bin/python3', '-u', 'scripts/kaggle_correctness_check.py']
[15:58:43] cwd=/kaggle/working/CayleyBeam100H100
[15:58:43] still_running | elapsed_sec=0.0 | silent_for_sec=0.0 | gpu=[0, Tesla T4, 3, 15360, 0 | 1, Tesla T4, 3, 15360, 0]


KeyboardInterrupt: 

In [ ]:
# 2-GPU correctness.
# Expected: final JSON from each rank. At least one global csv_step.remote_packed > 0 proves stream3/NCCL path was exercised.
if torch.cuda.device_count() >= 2:
    two_rank_env = FAST_ENV.copy()
    two_rank_env.update({
        'CUDA_VISIBLE_DEVICES': '0,1',
    })
    out_2gpu = run_live(
        [
            sys.executable,
            '-u',
            '-m',
            'torch.distributed.run',
            '--standalone',
            '--nnodes=1',
            '--nproc_per_node=2',
            'scripts/kaggle_correctness_check.py',
        ],
        env=two_rank_env,
        cwd=PROJECT_DIR,
        title='2-GPU correctness: stream1/stream2/stream3 + NCCL + CUDA Graph',
        heartbeat_sec=15,
        timeout_sec=900,
    )
else:
    out_2gpu = ''
    print('skip_2gpu=True; reason=less_than_2_cuda_devices_visible')


In [9]:
import subprocess

subprocess.run("pkill -f kaggle_correctness_check.py || true", shell=True)
subprocess.run("pkill -f torch.distributed.run || true", shell=True)
subprocess.run("pkill -f torchrun || true", shell=True)

print("old processes killed")

old processes killed


In [ ]:
# Optional: MLP/TorchScript ensemble quick check after the basic stream correctness passes.
# Stream correctness itself uses central_hamming first.
# TorchScript scorer paths are exported inside scripts/kaggle_correctness_check.py.

RUN_MLP_ENSEMBLE = True

if RUN_MLP_ENSEMBLE:
    mlp_env = FAST_ENV.copy()
    mlp_env.update({
        "WORLD_SIZE": "1",
        "RANK": "0",
        "LOCAL_RANK": "0",
        "CUDA_VISIBLE_DEVICES": "0",

        # IMPORTANT:
        # Do NOT set INFERENCE_BACKEND=torchscript_ensemble here.
        # The script first builds a base central_hamming engine,
        # then internally exports TorchScript copies and runs the ensemble case.
        "INFERENCE_BACKEND": "central_hamming",
        "TEST_TORCHSCRIPT_ENSEMBLE": "1",
        "INFERENCE_PARALLELISM": "2",

        "GLOBAL_BEAM_WIDTH": "512",
        "B_MICRO": "64",
        "SCORE_RING_DEPTH": "2",
        "NET_RING_DEPTH": "2",
        "BUCKET_CAP_PER_PEER": "2048",
        "MAX_DEPTH": "2",

        "BUILD_VERBOSE": "0",
        "EXT_VERBOSE": "0",
    })

    out_mlp = run_live(
        [sys.executable, "-u", "scripts/kaggle_correctness_check.py"],
        env=mlp_env,
        cwd=PROJECT_DIR,
        title="Optional 1-GPU TorchScript MLP ensemble correctness",
        heartbeat_sec=15,
        timeout_sec=900,
    )
else:
    print("skip_mlp_ensemble=True; set RUN_MLP_ENSEMBLE=True after base correctness passes")

In [10]:
import os
import sys
import time
import json
import queue
import signal
import subprocess
import threading
import textwrap
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/CayleyBeam100H100")
assert PROJECT_DIR.exists(), PROJECT_DIR

def now():
    return time.strftime("%H:%M:%S")

def gpu_snapshot():
    try:
        out = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
                "--format=csv,noheader,nounits",
            ],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
        return " | ".join(
            ", ".join(x.strip() for x in line.split(",")[:5])
            for line in out.splitlines()
        )
    except Exception as e:
        return f"nvidia-smi-error={type(e).__name__}:{e}"

def run_live(cmd, env, cwd, title, heartbeat_sec=2, timeout_sec=900):
    print("\n" + "=" * 96, flush=True)
    print(f"[{now()}] {title}", flush=True)
    print("=" * 96, flush=True)
    print(f"[{now()}] cmd={cmd}", flush=True)
    print(f"[{now()}] cwd={cwd}", flush=True)

    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        start_new_session=True,
    )

    q = queue.Queue()
    lines = []

    def reader():
        try:
            for line in proc.stdout:
                q.put(line)
        finally:
            q.put(None)

    threading.Thread(target=reader, daemon=True).start()

    t0 = time.time()
    last_output = t0
    last_heartbeat = 0.0

    while True:
        drained = False
        while True:
            try:
                item = q.get_nowait()
            except queue.Empty:
                break
            if item is None:
                continue
            drained = True
            last_output = time.time()
            lines.append(item.rstrip("\n"))
            print(item, end="", flush=True)

        rc = proc.poll()
        elapsed = time.time() - t0
        silent = time.time() - last_output

        if elapsed > timeout_sec and rc is None:
            print(f"[{now()}] timeout_kill | elapsed_sec={elapsed:.1f} | gpu=[{gpu_snapshot()}]", flush=True)
            try:
                os.killpg(proc.pid, signal.SIGKILL)
                print(f"[{now()}] timeout_kill_process_group | pid={proc.pid}", flush=True)
            except Exception as e:
                print(f"[{now()}] timeout_kill_process_group_failed | error={type(e).__name__}:{e}", flush=True)
                proc.kill()
            raise TimeoutError(f"timeout after {timeout_sec} sec")

        if rc is None and time.time() - last_heartbeat >= heartbeat_sec:
            print(
                f"[{now()}] still_running | elapsed_sec={elapsed:.1f} | "
                f"silent_for_sec={silent:.1f} | gpu=[{gpu_snapshot()}]",
                flush=True,
            )
            last_heartbeat = time.time()

        if rc is not None:
            while True:
                try:
                    item = q.get_nowait()
                except queue.Empty:
                    break
                if item is None:
                    continue
                lines.append(item.rstrip("\n"))
                print(item, end="", flush=True)

            print(f"[{now()}] process_exit | returncode={rc} | elapsed_sec={elapsed:.1f}", flush=True)
            text = "\n".join(lines)
            if rc != 0:
                raise subprocess.CalledProcessError(rc, cmd, output=text)
            return text

        time.sleep(0.2 if drained else 0.5)

def parse_step_summaries(text):
    rows = []
    for line in text.splitlines():
        if "STEP_SUMMARY " in line:
            payload = line.split("STEP_SUMMARY ", 1)[1].strip()
            try:
                rows.append(json.loads(payload))
            except Exception:
                pass
    return rows

# ---------------------------------------------------------------------
# 0. Kill stale distributed jobs.
# ---------------------------------------------------------------------

for pat in [
    "full_2gpu_algorithm_check_shared20m.py",
    "full_2gpu_algorithm_check.py",
    "heavy_2gpu_mlp_check.py",
    "kaggle_correctness_check.py",
    "torch.distributed.run",
    "torchrun",
]:
    subprocess.run(f"pkill -f {pat} || true", shell=True)

print(f"[{now()}] stale processes killed", flush=True)

# ---------------------------------------------------------------------
# 1. Write one 2-GPU full correctness runner:
#    1 TorchScript 20M-parameter model per rank.
#    4 inference lanes via INFERENCE_PARALLELISM=4.
#    Same scorer path repeated 4 times.
# ---------------------------------------------------------------------

runtime_dir = PROJECT_DIR / "runtime"
runtime_dir.mkdir(exist_ok=True)
runner = runtime_dir / "full_2gpu_algorithm_check_shared20m.py"

runner.write_text(textwrap.dedent(r'''
    from __future__ import annotations

    import json
    import os
    import sys
    from pathlib import Path

    import torch
    import torch.distributed as dist

    PROJECT_DIR = Path(__file__).resolve().parents[1]
    sys.path.insert(0, str(PROJECT_DIR))

    import data_loader
    import beam_engine
    from scorers import make_default_mlp_scorer


    def assert_true(cond: bool, msg: str) -> None:
        if not cond:
            raise AssertionError(msg)


    def allreduce_i64(values: list[int], device: torch.device) -> list[int]:
        t = torch.tensor(values, dtype=torch.int64, device=device)
        if dist.is_available() and dist.is_initialized():
            dist.all_reduce(t, op=dist.ReduceOp.SUM)
        return [int(x) for x in t.cpu().tolist()]


    def count_parameters(model: torch.nn.Module) -> int:
        return int(sum(p.numel() for p in model.parameters()))


    def export_one_torchscript_scorer_repeated_paths(rank: int, lanes: int) -> tuple[list[str], int]:
        """
        Python-level contract:
        - one PyTorch model object is constructed;
        - one TorchScript file is saved;
        - one path is repeated `lanes` times.

        Backend-level caveat:
        - if C++ torchscript_ensemble loads every path independently, then repeated paths
          can still become repeated torch::jit::Module objects.
        - true 1 weight-copy + 4 lanes requires backend-side shared Module ownership.
        """
        out_dir = PROJECT_DIR / "runtime" / "full_check_scorers_shared20m" / f"rank{rank}"
        out_dir.mkdir(parents=True, exist_ok=True)

        example = torch.zeros((16, 120), dtype=torch.uint8)

        torch.manual_seed(11000 + rank)
        model = make_default_mlp_scorer(
            hidden_sizes=(4096, 5120),
            fanout=24,
        ).eval()

        n_params = count_parameters(model)

        traced = torch.jit.trace(model, example, strict=False)
        traced = torch.jit.freeze(traced)

        path = out_dir / f"mlp_rank{rank}_single_shared_20m.ts"
        traced.save(str(path))

        paths = [str(path)] * lanes
        return paths, n_params


    def main() -> None:
        os.environ.setdefault("USE_CUDA_GRAPHS", "1")
        os.environ.setdefault("INFERENCE_BACKEND", "torchscript_ensemble")
        os.environ.setdefault("INFERENCE_PARALLELISM", "4")
        os.environ.setdefault("K_EXPAND_TILE", "8192")

        os.environ.setdefault("GLOBAL_BEAM_WIDTH", "2097152")
        os.environ.setdefault("B_MICRO", "8192")
        os.environ.setdefault("SCORE_RING_DEPTH", "8")
        os.environ.setdefault("NET_RING_DEPTH", "2")
        os.environ.setdefault("BUCKET_CAP_PER_PEER", "8192")
        os.environ.setdefault("BETA", "1.20")
        os.environ.setdefault("MAX_DEPTH", "8")
        os.environ.setdefault("HISTOGRAM_PERIOD_MICRO", "1")
        os.environ.setdefault("MLP_SCRAMBLE", "U,R,F,L,B,D,BR,DL")

        os.environ.setdefault("NCCL_IB_DISABLE", "1")
        os.environ.setdefault("NCCL_P2P_DISABLE", "1")
        os.environ.setdefault("NCCL_SOCKET_IFNAME", "lo")
        os.environ.setdefault("GLOO_SOCKET_IFNAME", "lo")

        cfg = beam_engine.make_default_config()
        beam_engine.init_distributed_if_needed(cfg)

        local_rank = int(os.environ.get("LOCAL_RANK", "0"))
        torch.cuda.set_device(local_rank)
        device = torch.device("cuda", local_rank)

        lanes = int(os.environ.get("INFERENCE_PARALLELISM", "4"))
        paths, n_params = export_one_torchscript_scorer_repeated_paths(cfg["rank"], lanes)
        unique_paths = sorted(set(paths))

        cfg["inference_backend"] = "torchscript_ensemble"
        cfg["torchscript_scorer_paths"] = os.pathsep.join(paths)
        cfg["inference_parallelism"] = lanes

        if cfg["rank"] == 0:
            print(
                "SCORER_CONFIG "
                + json.dumps(
                    {
                        "backend": cfg["inference_backend"],
                        "lanes_per_rank": lanes,
                        "scorer_params_per_rank": n_params,
                        "unique_scorer_paths_per_rank": len(unique_paths),
                        "path_repetition_per_rank": len(paths),
                        "hidden_sizes": [4096, 5120],
                        "fanout": 24,
                        "note": (
                            "Python exports one TorchScript file per rank and repeats one path 4 times. "
                            "True shared weights require backend not to torch::jit::load the same path 4 times."
                        ),
                    },
                    ensure_ascii=False,
                ),
                flush=True,
            )

        data_loader.validate_inverse_pairs()

        ext = beam_engine.build_extension(verbose=os.environ.get("BUILD_VERBOSE", "0") == "1")
        buffers = beam_engine.allocate_buffers(ext, cfg)
        engine = beam_engine.configure_engine(ext, cfg, buffers)

        scramble = [x.strip() for x in os.environ["MLP_SCRAMBLE"].split(",") if x.strip()]
        max_depth = int(os.environ["MAX_DEPTH"])

        state = data_loader.apply_actions_cpu(data_loader.get_central_state_u8(), scramble)
        owner = data_loader.owner_rank_for_state(state, cfg["world_size"])
        local_active = cfg["rank"] == owner

        engine.reset_search(state.tobytes(), local_active)

        step_reports = []
        final_depth = 0

        for depth in range(0, max_depth + 1):
            if depth > 0:
                engine.step(histogram_period_micro=cfg["histogram_period_micro"])

            st = dict(engine.status())
            counters = [int(x) for x in st["counters"]]

            # Counter convention observed in existing JSON:
            # [next_pool_size, local_inserted, local_updated, remote_packed,
            #  bucket_overflow, hash_overflow, pruned, reserved/debug]
            sums = allreduce_i64(
                [
                    int(st["found"]),
                    int(st["current_size"]),
                    int(st["compacted_size"]),
                    counters[0],
                    counters[1],
                    counters[2],
                    counters[3],
                    counters[4],
                    counters[5],
                    counters[6],
                    counters[7],
                    int(st["cuda_graph_captured"]),
                    int(st["threshold_valid"]),
                    int(st["threshold_q"]),
                    n_params,
                    len(unique_paths),
                    len(paths),
                ],
                device,
            )

            row = {
                "depth": depth,
                "global_found": sums[0],
                "global_current_size": sums[1],
                "global_compacted_size": sums[2],
                "global_next_pool_size": sums[3],
                "global_local_inserted": sums[4],
                "global_local_updated": sums[5],
                "global_remote_packed": sums[6],
                "global_bucket_overflow": sums[7],
                "global_hash_overflow": sums[8],
                "global_pruned": sums[9],
                "global_counter7": sums[10],
                "cuda_graph_captured_sum": sums[11],
                "threshold_valid_sum": sums[12],
                "threshold_q_sum": sums[13],
                "global_scorer_params": sums[14],
                "global_unique_scorer_paths": sums[15],
                "global_path_repetitions": sums[16],
                "rank": cfg["rank"],
                "local_current_size": int(st["current_size"]),
                "local_found": int(st["found"]),
                "local_counters": counters,
            }

            step_reports.append(row)
            final_depth = depth

            if cfg["rank"] == 0:
                print("STEP_SUMMARY", json.dumps(row, ensure_ascii=False), flush=True)

            assert_true(sums[7] == 0, f"bucket_overflow at depth={depth}: {sums[7]}")
            assert_true(sums[8] == 0, f"hash_overflow at depth={depth}: {sums[8]}")

            if sums[0] > 0:
                break

        final_status = dict(engine.status())
        final_counters = [int(x) for x in final_status["counters"]]

        final_sums = allreduce_i64(
            [
                int(final_status["found"]),
                int(final_status["current_size"]),
                final_counters[0],
                final_counters[1],
                final_counters[2],
                final_counters[3],
                final_counters[4],
                final_counters[5],
                final_counters[6],
                int(final_status["cuda_graph_captured"]),
                n_params,
                len(unique_paths),
                len(paths),
            ],
            device,
        )

        report = {
            "rank": cfg["rank"],
            "world_size": cfg["world_size"],
            "local_rank": local_rank,
            "device": torch.cuda.get_device_name(device),
            "backend": cfg["inference_backend"],
            "lanes_per_rank": lanes,
            "scorer_params_per_rank": n_params,
            "unique_scorer_paths_per_rank": len(unique_paths),
            "path_repetition_per_rank": len(paths),
            "paths": paths,
            "unique_paths": unique_paths,
            "scramble": scramble,
            "owner": owner,
            "max_depth": max_depth,
            "final_depth": final_depth,
            "config": {
                "GLOBAL_BEAM_WIDTH": int(os.environ["GLOBAL_BEAM_WIDTH"]),
                "B_MICRO": int(os.environ["B_MICRO"]),
                "SCORE_RING_DEPTH": int(os.environ["SCORE_RING_DEPTH"]),
                "NET_RING_DEPTH": int(os.environ["NET_RING_DEPTH"]),
                "BUCKET_CAP_PER_PEER": int(os.environ["BUCKET_CAP_PER_PEER"]),
                "INFERENCE_PARALLELISM": lanes,
                "K_EXPAND_TILE": int(os.environ.get("K_EXPAND_TILE", "0")),
                "BETA": float(os.environ.get("BETA", "0")),
                "HISTOGRAM_PERIOD_MICRO": int(os.environ["HISTOGRAM_PERIOD_MICRO"]),
                "HIDDEN_SIZES": [4096, 5120],
                "FANOUT": 24,
            },
            "step_reports": step_reports,
            "local_status": final_status,
            "global_sums": {
                "found": final_sums[0],
                "current_size": final_sums[1],
                "next_pool_size": final_sums[2],
                "local_inserted": final_sums[3],
                "local_updated": final_sums[4],
                "remote_packed": final_sums[5],
                "bucket_overflow": final_sums[6],
                "hash_overflow": final_sums[7],
                "pruned": final_sums[8],
                "cuda_graph_captured_sum": final_sums[9],
                "scorer_params_sum": final_sums[10],
                "unique_scorer_paths_sum": final_sums[11],
                "path_repetition_sum": final_sums[12],
            },
        }

        if cfg["rank"] == 0:
            print("FULL_ALGORITHM_CORRECTNESS_FINISHED", json.dumps(report["global_sums"], ensure_ascii=False), flush=True)

        print(json.dumps(report, indent=2, ensure_ascii=False), flush=True)

        # Final correctness assertions.
        assert_true(cfg["world_size"] == 2, "expected world_size=2")
        assert_true(report["backend"] == "torchscript_ensemble", "expected torchscript_ensemble backend")
        assert_true(report["lanes_per_rank"] == 4, "expected 4 inference lanes per rank")
        assert_true(report["unique_scorer_paths_per_rank"] == 1, "expected one scorer file per rank")
        assert_true(report["path_repetition_per_rank"] == 4, "expected same scorer path repeated 4 times per rank")
        assert_true(report["scorer_params_per_rank"] >= 20_000_000, "expected >=20M scorer parameters per rank")
        assert_true(report["global_sums"]["cuda_graph_captured_sum"] >= 2, "CUDA Graph not captured on both ranks")
        assert_true(report["global_sums"]["local_inserted"] + report["global_sums"]["remote_packed"] > 0, "zero generated candidates")
        assert_true(report["global_sums"]["remote_packed"] > 0, "Stream3/NCCL remote path not exercised")
        assert_true(report["global_sums"]["bucket_overflow"] == 0, "bucket overflow")
        assert_true(report["global_sums"]["hash_overflow"] == 0, "hash overflow")
        assert_true(max(x["global_current_size"] for x in step_reports if x["depth"] > 0) > 0, "beam did not grow")

        sys.stdout.flush()
        sys.stderr.flush()
        os._exit(0)


    if __name__ == "__main__":
        try:
            main()
        except BaseException as exc:
            print(f"FATAL_EXIT type={type(exc).__name__}; message={exc}", flush=True)
            sys.stdout.flush()
            sys.stderr.flush()
            os._exit(1)
'''))

# ---------------------------------------------------------------------
# 2. Run config.
# ---------------------------------------------------------------------

env = os.environ.copy()
env.update({
    "PYTHONUNBUFFERED": "1",
    "CUDA_VISIBLE_DEVICES": "0,1",

    "USE_CUDA_GRAPHS": "1",
    "INFERENCE_BACKEND": "torchscript_ensemble",

    # 4 inference lanes.
    # Python-level scorer export below creates one .ts file per rank and repeats path 4 times.
    "INFERENCE_PARALLELISM": "4",
    "K_EXPAND_TILE": "8192",

    # First stable 20M-MLP config.
    # After pass: increase GLOBAL_BEAM_WIDTH/MAX_DEPTH separately.
    "GLOBAL_BEAM_WIDTH": "2097152",
    "B_MICRO": "8192",
    "SCORE_RING_DEPTH": "8",
    "NET_RING_DEPTH": "2",
    "BUCKET_CAP_PER_PEER": "8192",
    "BETA": "1.20",
    "HASH_LOAD_FACTOR": "0.45",
    "PROBE_LIMIT": "128",
    "MAX_DEPTH": "8",
    "HISTOGRAM_PERIOD_MICRO": "1",

    # Only known action names.
    "MLP_SCRAMBLE": "U,R,F,L,B,D,BR,DL",

    "BUILD_VERBOSE": "0",
    "EXT_VERBOSE": "0",
    "NCCL_DEBUG": "WARN",

    "NCCL_IB_DISABLE": "1",
    "NCCL_P2P_DISABLE": "1",
    "NCCL_SOCKET_IFNAME": "lo",
    "GLOO_SOCKET_IFNAME": "lo",
})

out = run_live(
    [
        sys.executable, "-u", "-m", "torch.distributed.run",
        "--standalone",
        "--nnodes=1",
        "--nproc_per_node=2",
        str(runner),
    ],
    env=env,
    cwd=PROJECT_DIR,
    title="ONE-CELL SHARED-SCORER TEST: 2-GPU + 1x20M TorchScript path/rank + 4 inference lanes + CUDA Graph + NCCL",
    heartbeat_sec=2,
    timeout_sec=900,
)

assert "FULL_ALGORITHM_CORRECTNESS_FINISHED" in out, "final success marker not found"

steps = parse_step_summaries(out)
assert len(steps) > 0, "STEP_SUMMARY not found"

print("\nPER_STEP_BEAM_TABLE", flush=True)
print(
    "depth | found | current | next_pool | inserted | updated | remote | bucket_ovf | hash_ovf | pruned | graph_sum | th_valid_sum | th_q_sum | params_sum | unique_paths | path_reps",
    flush=True,
)
for r in steps:
    print(
        f"{r['depth']:>5} | "
        f"{r['global_found']:>5} | "
        f"{r['global_current_size']:>7} | "
        f"{r['global_next_pool_size']:>9} | "
        f"{r['global_local_inserted']:>8} | "
        f"{r['global_local_updated']:>7} | "
        f"{r['global_remote_packed']:>6} | "
        f"{r['global_bucket_overflow']:>10} | "
        f"{r['global_hash_overflow']:>8} | "
        f"{r['global_pruned']:>6} | "
        f"{r['cuda_graph_captured_sum']:>9} | "
        f"{r['threshold_valid_sum']:>12} | "
        f"{r['threshold_q_sum']:>8} | "
        f"{r['global_scorer_params']:>10} | "
        f"{r['global_unique_scorer_paths']:>12} | "
        f"{r['global_path_repetitions']:>9}",
        flush=True,
    )

verdict = {
    "two_gpu": True,
    "one_unique_scorer_path_per_rank": all(r["global_unique_scorer_paths"] == 2 for r in steps),
    "four_path_repetitions_per_rank": all(r["global_path_repetitions"] == 8 for r in steps),
    "twenty_m_params_per_rank": all(r["global_scorer_params"] >= 40_000_000 for r in steps),
    "four_inference_lanes_requested": True,
    "cuda_graph_on_both_ranks": any(r["cuda_graph_captured_sum"] >= 2 for r in steps if r["depth"] > 0),
    "beam_grew": max(r["global_current_size"] for r in steps if r["depth"] > 0) > 0,
    "stream3_nccl_used": max(r["global_remote_packed"] for r in steps) > 0,
    "bucket_overflow_zero_all_steps": all(r["global_bucket_overflow"] == 0 for r in steps),
    "hash_overflow_zero_all_steps": all(r["global_hash_overflow"] == 0 for r in steps),
}

print("\nFULL_CORRECTNESS_VERDICT", json.dumps(verdict, indent=2, ensure_ascii=False), flush=True)
assert all(verdict.values()), verdict

print(
    "\nFULL_ALGORITHM_OK: 2-GPU + one 20M TorchScript scorer path per rank + 4 requested inference lanes + CUDA Graph + Stream1/2/3 + NCCL + per-step beam accounting verified.",
    flush=True,
)

print(
    "\nBACKEND_CAVEAT: this cell guarantees one exported TorchScript file per rank and four repeated scorer paths. "
    "If C++ torchscript_ensemble loads repeated paths as separate torch::jit::Module objects, real VRAM weights are still copied. "
    "True 1-weight-copy + 4-lane execution then requires backend-side shared module ownership.",
    flush=True,
)

[15:59:03] stale processes killed

[15:59:03] ONE-CELL SHARED-SCORER TEST: 2-GPU + 1x20M TorchScript path/rank + 4 inference lanes + CUDA Graph + NCCL
[15:59:03] cmd=['/usr/bin/python3', '-u', '-m', 'torch.distributed.run', '--standalone', '--nnodes=1', '--nproc_per_node=2', '/kaggle/working/CayleyBeam100H100/runtime/full_2gpu_algorithm_check_shared20m.py']
[15:59:03] cwd=/kaggle/working/CayleyBeam100H100
[15:59:03] still_running | elapsed_sec=0.0 | silent_for_sec=0.0 | gpu=[0, Tesla T4, 3, 15360, 0 | 1, Tesla T4, 3, 15360, 0]
[15:59:05] still_running | elapsed_sec=2.0 | silent_for_sec=2.0 | gpu=[0, Tesla T4, 3, 15360, 0 | 1, Tesla T4, 3, 15360, 0]

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[W504 15:59:05.880224018 socket.cpp:207] [c10d] 

KeyboardInterrupt: 